# 1. Neural News Recommendation with Multi-Head Self-Attention (NRMS)

A text-only NRMS reranker. It learns a user representation from the articles the user
clicked before the impression and scores each candidate by its similarity to that user.

- **News encoder**: the frozen 64-dimensional text embedding of an article, followed by a
  shared linear projection and layer normalisation
- **User encoder**:
    - **Self-attention block**: multi-head self-attention lets each clicked article attend
      to the others, followed by a residual connection and layer normalisation
    - **Additive attention**: learns a relevance weight for every clicked article and pools
      the sequence into one user vector; users with no history get a learned default vector
- **Click predictor**: the scaled dot product between the user vector and the candidate vector

The notebook has two experiments:

1. **Architecture search**: attention heads × dropout, trained with weighted BCE
2. **Loss ablation**: the selected architecture trained with BCE, BPR and BCE–BPR hybrids

The single best validation trial across both experiments is evaluated on test.

## 1.1 Load Packages

In [ ]:
import json
import random
import sys
from itertools import cycle
from pathlib import Path

import numpy as np
import pandas as pd
import polars as pl
import pyarrow.parquet as pq
import torch
import torch.nn.functional as F
from IPython.display import display
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

## 1.2 Set Up Paths

Set `ENVIRONMENT` to `"colab"` to read the project and data from Google Drive. The
repository supplies the shared evaluator; the data folder holds the prepared features
and receives every checkpoint, score file and metric table this notebook writes.

In [ ]:
ENVIRONMENT = "local"  # "local" or "colab"

if ENVIRONMENT == "colab":
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_ROOT = Path("/content/drive/MyDrive/BT4222-Group-6-")
    DATA_ROOT = Path("/content/drive/MyDrive/v3-mindsmall")
else:
    PROJECT_ROOT = Path("/Users/numkoos/Downloads/BT4222-Group-6-")
    DATA_ROOT = Path("/Users/numkoos/Downloads/v3-mindsmall")

# Ranking features, click labels and causal user histories from feature engineering
RUN = DATA_ROOT / "4_Feature_Engineering" / "artifacts" / "prequential-all"

# Frozen article embeddings from the representation stage
REPRESENTATIONS = DATA_ROOT / "3_Representations" / "artifacts"

# Checkpoints, score files and metric tables written by the reranking notebooks
ARTIFACTS = DATA_ROOT / "5_Models" / "5_02_Reranking" / "artifacts"
RESULTS = DATA_ROOT / "results"

# The shared evaluator is the only project module the reranking notebooks import
sys.path.insert(0, str(PROJECT_ROOT / "5_Models" / "5_02_Reranking"))
from rr_evaluation import compare_results, rank_logged_slate, score_and_evaluate

## 1.3 Load Ranking Features

Each split is stored as Parquet partitions with one row per (impression, candidate)
pair. These previews read only the first rows; later cells load what they need.

In [ ]:
def preview_split(split, n_rows=5):
    """Read the first rows of a split without loading every partition."""
    first_part = sorted((RUN / f"ranking_features_{split}").glob("*.parquet"))[0]
    return next(pq.ParquetFile(first_part).iter_batches(batch_size=n_rows)).to_pandas()


train_preview = preview_split("train")
validation_preview = preview_split("validation")
train_preview.head()

In [ ]:
validation_preview.head()

# 2. Experiment Configuration

The architecture search trains on a uniform row sample; the loss ablation trains on whole
sampled impressions because BPR needs a click and its unclicked neighbours together.

In [ ]:
SEED = 4222
DEVICE = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
print(f"Using {DEVICE}")

HISTORY_CAP = 50
MODEL_DIM = 64
EPOCHS = 5
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-5
SELECTION_METRIC = "ndcg_at_10"
BATCH_SIZE = 2048   # scoring batch size; training batch sizes are set per experiment

# Experiment 1: architecture search
ARCH_TRAIN_ROWS = 300_000
ARCH_BATCH_SIZE = 2048
ARCH_CONFIGS = [
    {"model_dim": MODEL_DIM, "heads": heads, "dropout": dropout}
    for heads in (1, 2, 4)
    for dropout in (0.0, 0.1, 0.2)
]
RESUME = True   # reuse trials whose checkpoint and validation metrics already exist

# Experiment 2: loss and negative-sampling ablation
LOSS_TRAIN_IMPRESSIONS = 10_000
LOSS_BATCH_SIZE = 256
NEGATIVES_PER_POSITIVE = 4
POPULARITY_ALPHAS = (0.0, 0.5, 1.0)
HYBRID_WEIGHTS = (0.25, 0.50, 0.75)
LOSS_OUTPUT_DIR = ARTIFACTS / "pure_nrms_loss_ablation"

# 3. Prepare Training Data

## 3.1 Sample Labelled Training Candidates

`sample_train_rows` draws a reproducible, uniform sample of training candidates. The
number of rows taken from each partition follows a hypergeometric draw, so the whole
training split never has to be loaded at once.

In [ ]:
def read_labelled_part(part, columns):
    """Join one training feature partition to its click labels."""
    features = pq.read_table(part, columns=["impression_uid", "news_code", *columns]).to_pandas()
    labels = pq.read_table(
        RUN / "labels_train" / part.name,
        columns=["impression_uid", "news_code", "clicked"],
    ).to_pandas()
    return features.merge(labels, on=["impression_uid", "news_code"], validate="one_to_one")


def sample_train_rows(limit, columns):
    """Sample `limit` labelled training candidates uniformly across all partitions."""
    parts = sorted((RUN / "ranking_features_train").glob("*.parquet"))
    sizes = [pq.ParquetFile(part).metadata.num_rows for part in parts]
    rng = np.random.default_rng(SEED)
    remaining_population = sum(sizes)
    remaining_take = min(limit, remaining_population)

    samples = []
    for part, size in zip(parts, sizes):
        take = int(rng.hypergeometric(size, remaining_population - size, remaining_take))
        if take:
            frame = read_labelled_part(part, columns)
            samples.append(frame.iloc[rng.choice(size, take, replace=False)])
        remaining_population -= size
        remaining_take -= take
    return pd.concat(samples, ignore_index=True)

## 3.2 Sample Training Impressions and Preference Pairs

Used by the loss ablation. Prior exposures only weight negative sampling; they are not a
model input.

In [ ]:
def sample_train_impressions(max_impressions, columns):
    """Sample whole training impressions, then load every retained candidate in them.

    Keeping impressions whole lets BPR pair each click with unclicked candidates from
    the same slate. Feature engineering already sampled training negatives, so these
    are the retained candidates rather than the complete displayed slate.
    """
    feature_files = sorted((RUN / "ranking_features_train").glob("*.parquet"))
    label_files = sorted((RUN / "labels_train").glob("*.parquet"))
    impression_ids = (
        pl.scan_parquet(label_files)
        .select("impression_uid").unique().sort("impression_uid")
        .collect()["impression_uid"].to_numpy()
    )
    rng = np.random.default_rng(SEED)
    chosen = rng.choice(impression_ids, size=min(max_impressions, len(impression_ids)), replace=False)

    keys = ["impression_uid", "news_code"]
    read_columns = list(dict.fromkeys([*keys, "prior_article_exposures", *columns]))
    features = (
        pl.scan_parquet(feature_files)
        .filter(pl.col("impression_uid").is_in(chosen))
        .select(read_columns).collect().to_pandas()
    )
    labels = (
        pl.scan_parquet(label_files)
        .filter(pl.col("impression_uid").is_in(chosen))
        .select(*keys, "clicked").collect().to_pandas()
    )
    train = features.merge(labels, on=keys, how="inner", validate="one_to_one")
    if not len(train) == len(features) == len(labels):
        raise ValueError("Training feature and label keys do not match")
    return train.sort_values(keys).reset_index(drop=True)

In [ ]:
def sample_pairs(train, *, popularity_alpha):
    """Pair each click with up to NEGATIVES_PER_POSITIVE unclicked candidates from its impression.

    Negatives are drawn without replacement with weight (1 + prior exposures) ** alpha:
    alpha = 0 is uniform and larger alpha favours popular articles. Impressions without
    an unclicked candidate contribute no pairs.
    """
    rng = np.random.default_rng(SEED)
    clicked = train["clicked"].to_numpy()
    popularity = train["prior_article_exposures"].to_numpy(dtype=float)

    positive_rows, negative_rows = [], []
    for rows in train.groupby("impression_uid", sort=False).indices.values():
        positives = rows[clicked[rows] == 1]
        negatives = rows[clicked[rows] == 0]
        if not len(negatives):
            continue
        # Work in log space so very popular articles cannot overflow the weights
        log_weights = popularity_alpha * np.log1p(popularity[negatives])
        weights = np.exp(log_weights - log_weights.max())
        probabilities = weights / weights.sum()
        for positive in positives:
            size = min(NEGATIVES_PER_POSITIVE, len(negatives))
            sampled = rng.choice(negatives, size=size, replace=False, p=probabilities)
            positive_rows.extend([positive] * len(sampled))
            negative_rows.extend(sampled.tolist())
    return np.asarray(positive_rows, dtype=np.int64), np.asarray(negative_rows, dtype=np.int64)

## 3.3 Article Vectors and Causal Click Histories

A user's history is the list of articles they clicked **strictly before** the impression;
at most the newest `HISTORY_CAP` clicks are used.

In [ ]:
# Map each news_code to its row in the frozen 64-dimensional text embedding matrix.
# Row 0 is reserved for padding.
article_index = pl.read_parquet(REPRESENTATIONS / "article_index.parquet", columns=["news_code", "article_row"])
news_to_row = dict(zip(article_index["news_code"].to_list(), article_index["article_row"].to_list()))
text_vectors = np.load(REPRESENTATIONS / "text64.npy", mmap_mode="r")
EMBEDDING_DIM = text_vectors.shape[1]
print(f"Article embeddings: {text_vectors.shape[0]:,} rows x {EMBEDDING_DIM} dimensions")

In [ ]:
def load_profiles(split, impression_ids=None):
    """Map each impression to the article rows its user clicked strictly before the impression."""
    profiles = (
        pl.scan_parquet(RUN / f"request_profiles_{split}" / "*.parquet")
        .select("impression_uid", "full_history_rows")
    )
    if impression_ids is not None:
        profiles = profiles.filter(pl.col("impression_uid").is_in(list(impression_ids)))
    profiles = profiles.collect()
    return dict(zip(profiles["impression_uid"].to_list(), profiles["full_history_rows"].to_list()))

In [ ]:
def encode_pairs(frame, profiles):
    """Return the padded click history and the candidate article row for every candidate.

    Histories keep the newest HISTORY_CAP clicks and are left-padded with row 0.
    """
    histories = np.zeros((len(frame), HISTORY_CAP), dtype=np.int64)
    candidates = np.empty(len(frame), dtype=np.int64)
    for index, (uid, news_code) in enumerate(zip(frame["impression_uid"], frame["news_code"])):
        history = profiles[uid][-HISTORY_CAP:]
        if len(history):
            histories[index, -len(history):] = history
        candidates[index] = news_to_row[news_code]
    return histories, candidates

# 4. Model

## 4.1 Additive Attention

After self-attention contextualises the clicked articles, additive attention scores each
one with a learned query vector and returns the weighted sum as the user vector. Padding
receives zero weight.

In [ ]:
class AdditiveAttention(nn.Module):
    """Pool a sequence of history states into one user vector with learned relevance weights."""

    def __init__(self, dim):
        super().__init__()
        self.proj = nn.Linear(dim, dim)
        self.query = nn.Parameter(torch.randn(dim) * 0.02)

    def forward(self, states, padding_mask):
        # states: [batch, history, dim]; padding_mask: [batch, history], True = padding
        all_padded = padding_mask.all(dim=1)
        safe_mask = padding_mask.clone()
        safe_mask[all_padded, 0] = False                       # softmax needs one open slot

        scores = torch.tanh(self.proj(states)) @ self.query    # [batch, history]
        weights = torch.softmax(scores.masked_fill(safe_mask, -1e9), dim=1)
        weights = weights.masked_fill(padding_mask, 0.0)
        weights = weights / weights.sum(dim=1, keepdim=True).clamp(min=1e-8)
        return (weights.unsqueeze(-1) * states).sum(dim=1)     # [batch, dim]

## 4.2 `PureNRMS`

```text
Clicked articles → text embedding → projection → multi-head self-attention
                 → residual + LayerNorm → additive attention → user vector
Candidate        → text embedding → projection → candidate vector
score = user · candidate / √dim
```

In [ ]:
class PureNRMS(nn.Module):
    """Frozen article text → history self-attention → additive pooling → dot-product score."""

    def __init__(self, vectors, model_dim=64, heads=2, dropout=0.1):
        super().__init__()
        vectors = torch.from_numpy(np.array(vectors, dtype=np.float32, copy=True))
        self.articles = nn.Embedding.from_pretrained(vectors, freeze=True, padding_idx=0)
        self.project = nn.Sequential(nn.Linear(vectors.shape[1], model_dim), nn.LayerNorm(model_dim))
        self.attention = nn.MultiheadAttention(model_dim, heads, dropout=dropout, batch_first=True)
        self.norm = nn.LayerNorm(model_dim)
        self.pool = AdditiveAttention(model_dim)
        self.default_user = nn.Parameter(torch.zeros(model_dim))   # used when there is no history

    def forward(self, history_rows, candidate_rows):
        padding_mask = history_rows.eq(0)                  # [batch, history]
        empty = padding_mask.all(dim=1)
        safe_mask = padding_mask.clone()
        safe_mask[empty, 0] = False

        history = self.project(self.articles(history_rows))       # [batch, history, dim]
        candidate = self.project(self.articles(candidate_rows))   # [batch, dim]

        attended, _ = self.attention(history, history, history, key_padding_mask=safe_mask, need_weights=False)
        states = self.norm(history + attended).masked_fill(padding_mask.unsqueeze(-1), 0.0)
        user = self.pool(states, padding_mask)
        user = torch.where(empty.unsqueeze(1), self.default_user.unsqueeze(0), user)
        return (user * candidate).sum(dim=1) / np.sqrt(candidate.shape[1])

# 5. Training

- **BCE** iterates over shuffled candidate batches with a training-derived positive weight
- **BPR** iterates over click/non-click pairs from the same impression: $-\log\sigma(s^+ - s^-)$
- **Hybrid** draws one candidate batch and one pair batch per step: $(1-w)\,\text{BCE} + w\,\text{BPR}$

In [ ]:
def train_nrms(config, histories, candidates, labels, *, batch_size,
               objective="bce", pairs=None, hybrid_weight=0.5):
    """Train a fresh PureNRMS with the BCE, BPR or hybrid objective."""
    history = torch.from_numpy(histories)
    candidate = torch.from_numpy(candidates)
    y = torch.from_numpy(labels)
    click_rate = labels.mean()
    positive_weight = torch.tensor([(1 - click_rate) / click_rate], device=DEVICE)

    torch.manual_seed(SEED)
    model = PureNRMS(text_vectors, **config).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

    row_loader = DataLoader(
        TensorDataset(torch.arange(len(y))),
        batch_size=batch_size,
        shuffle=True,
        generator=torch.Generator().manual_seed(SEED),
    )
    pair_loader = None
    if objective != "bce":
        pair_loader = DataLoader(
            TensorDataset(torch.from_numpy(pairs[0]), torch.from_numpy(pairs[1])),
            batch_size=batch_size,
            shuffle=True,
            generator=torch.Generator().manual_seed(SEED + 1),
        )

    def score(rows):
        return model(history[rows].to(DEVICE), candidate[rows].to(DEVICE))

    for epoch in range(EPOCHS):
        model.train()
        pair_batches = cycle(pair_loader) if pair_loader is not None else None
        losses = []
        for (rows,) in row_loader:
            optimizer.zero_grad()

            if objective in ("bce", "hybrid"):
                bce = F.binary_cross_entropy_with_logits(score(rows), y[rows].to(DEVICE), pos_weight=positive_weight)
            if objective in ("bpr", "hybrid"):
                positive, negative = next(pair_batches)
                bpr = -F.logsigmoid(score(positive) - score(negative)).mean()

            if objective == "bce":
                loss = bce
            elif objective == "bpr":
                loss = bpr
            else:
                loss = (1 - hybrid_weight) * bce + hybrid_weight * bpr

            loss.backward()
            optimizer.step()
            losses.append(loss.item())
        print(f"  epoch {epoch + 1}/{EPOCHS}: mean batch loss {np.mean(losses):.5f}")
    return model

In [ ]:
def predict_batches(model, *inputs):
    """Score aligned NumPy inputs in mini-batches and return click probabilities."""
    model.eval()
    scores = []
    with torch.no_grad():
        for start in range(0, len(inputs[0]), BATCH_SIZE):
            batch = [torch.from_numpy(array[start:start + BATCH_SIZE]).to(DEVICE) for array in inputs]
            scores.append(torch.sigmoid(model(*batch)).cpu().numpy())
    return np.concatenate(scores)

In [ ]:
def nrms_predictor(model, profiles):
    """Wrap a trained NRMS and a split's histories as a predict(frame) function."""
    def predict(frame):
        histories, candidates = encode_pairs(frame, profiles)
        return predict_batches(model, histories, candidates)
    return predict


def save_and_validate(model, config, name, output_dir, metadata):
    """Save a trained model and evaluate it on the complete validation slates."""
    output_dir.mkdir(parents=True, exist_ok=True)
    torch.save({"state_dict": model.state_dict(), "config": config}, output_dir / "model.pt")
    (output_dir / "metadata.json").write_text(json.dumps({"name": name, **metadata}, indent=2) + "\n")
    result, _ = score_and_evaluate(
        nrms_predictor(model, validation_profiles),
        split="validation",
        feature_run=RUN,
        output_dir=output_dir,
        model_name=name,
        results_dir=RESULTS,
    )
    return result


validation_profiles = load_profiles("validation")

# 6. Experiment 1: Architecture Search

Every combination of attention heads and dropout is trained with weighted BCE on the same
row sample. Completed trials are reloaded when `RESUME` is on.

In [ ]:
arch_train = sample_train_rows(ARCH_TRAIN_ROWS, columns=[])
arch_profiles = load_profiles("train", arch_train["impression_uid"].unique())
arch_histories, arch_candidates = encode_pairs(arch_train, arch_profiles)
arch_labels = arch_train["clicked"].to_numpy(np.float32)
print(f"Architecture search sample: {len(arch_train):,} candidates")


def arch_name(config):
    return f"pure_nrms_h{config['heads']}_d{config['dropout']:.1f}_v1"


arch_results = []
for config in ARCH_CONFIGS:
    name = arch_name(config)
    output_dir = ARTIFACTS / name
    metrics_path = RESULTS / f"reranking_{name}_validation.csv"
    if RESUME and metrics_path.exists() and (output_dir / "model.pt").exists():
        print(f"already complete: {name}")
        result = pl.read_csv(metrics_path)
    else:
        print(f"\nrunning: {name}")
        model = train_nrms(config, arch_histories, arch_candidates, arch_labels, batch_size=ARCH_BATCH_SIZE)
        result = save_and_validate(model, config, name, output_dir,
                                   {"objective": "bce", "train_rows": len(arch_train), "seed": SEED})
    arch_results.append(result.with_columns(
        pl.lit(config["heads"]).alias("heads"),
        pl.lit(config["dropout"]).alias("dropout"),
    ))

arch_leaderboard = compare_results(arch_results, sort_by=SELECTION_METRIC)
arch_leaderboard.write_csv(RESULTS / "pure_nrms_validation_leaderboard.csv")
best_arch = next(config for config in ARCH_CONFIGS if arch_name(config) == arch_leaderboard["model"][0])
print(f"Selected architecture: {best_arch}")
display(arch_leaderboard.select("model", "heads", "dropout", "mrr", "ndcg_at_10"))

# 7. Experiment 2: Loss and Negative-Sampling Ablation

The selected architecture is retrained on whole sampled impressions with:

1. BCE, and BPR with uniform or popularity-weighted negatives (best sampler chosen on validation)
2. BCE–BPR hybrids using the selected sampler

In [ ]:
loss_train = sample_train_impressions(LOSS_TRAIN_IMPRESSIONS, columns=[])
loss_profiles = load_profiles("train", loss_train["impression_uid"].unique())
loss_histories, loss_candidates = encode_pairs(loss_train, loss_profiles)
loss_labels = loss_train["clicked"].to_numpy(np.float32)
print(f"Loss ablation sample: {loss_train['impression_uid'].nunique():,} impressions, "
      f"{len(loss_train):,} candidates, click rate {loss_labels.mean():.3%}")


def run_loss_trial(name, objective, *, popularity_alpha=0.0, hybrid_weight=None):
    """Train, save and validate the selected architecture with one objective."""
    pairs = sample_pairs(loss_train, popularity_alpha=popularity_alpha)
    print(f"\n{name}: {objective}, popularity alpha={popularity_alpha}, pairs={len(pairs[0]):,}")
    model = train_nrms(best_arch, loss_histories, loss_candidates, loss_labels,
                       batch_size=LOSS_BATCH_SIZE, objective=objective, pairs=pairs,
                       hybrid_weight=hybrid_weight or 0.5)
    metadata = {"objective": objective, "popularity_alpha": popularity_alpha,
                "hybrid_weight": hybrid_weight, "training_rows": len(loss_train),
                "pair_count": len(pairs[0]), "seed": SEED}
    result = save_and_validate(model, best_arch, name, LOSS_OUTPUT_DIR / name, metadata)
    return result.with_columns(
        pl.lit(objective).alias("objective"),
        pl.lit(float(popularity_alpha)).alias("popularity_alpha"),
        pl.lit(hybrid_weight, dtype=pl.Float64).alias("hybrid_weight"),
    )


loss_results = [run_loss_trial("nrms_bce", "bce")]
for alpha in POPULARITY_ALPHAS:
    loss_results.append(run_loss_trial(f"nrms_bpr_pop_{alpha:.2f}", "bpr", popularity_alpha=alpha))

bpr_results = compare_results(loss_results, sort_by=SELECTION_METRIC).filter(pl.col("objective") == "bpr")
best_popularity_alpha = bpr_results["popularity_alpha"][0]
print(f"BPR sampling selected on validation: popularity alpha={best_popularity_alpha}")

In [ ]:
for weight in HYBRID_WEIGHTS:
    name = f"nrms_hybrid_bpr_{weight:.2f}_pop_{best_popularity_alpha:.2f}"
    loss_results.append(run_loss_trial(name, "hybrid", popularity_alpha=best_popularity_alpha, hybrid_weight=weight))

loss_leaderboard = compare_results(loss_results, sort_by=SELECTION_METRIC)
loss_leaderboard.write_csv(RESULTS / "reranking_nrms_loss_ablation_validation.csv")
display(loss_leaderboard.select(
    "model", "objective", "popularity_alpha", "hybrid_weight", "mrr", "ndcg_at_5", "ndcg_at_10",
))

# 8. Test Evaluation

The best validation trial across both experiments is reloaded and scored on test.

In [ ]:
final = compare_results([arch_leaderboard, loss_leaderboard], sort_by=SELECTION_METRIC).row(0, named=True)
final_model_name = final["model"]
final_dir = ARTIFACTS / final_model_name if final_model_name.startswith("pure_nrms_h") else LOSS_OUTPUT_DIR / final_model_name
print(f"Validation-selected trial: {final_model_name} ({SELECTION_METRIC}={final[SELECTION_METRIC]:.4f})")

checkpoint = torch.load(final_dir / "model.pt", map_location=DEVICE, weights_only=False)
final_model = PureNRMS(text_vectors, **checkpoint["config"]).to(DEVICE)
final_model.load_state_dict(checkpoint["state_dict"])

test_result, _ = score_and_evaluate(
    nrms_predictor(final_model, load_profiles("test")),
    split="test",
    feature_run=RUN,
    output_dir=final_dir,
    model_name=final_model_name,
    results_dir=RESULTS,
)
display(test_result.select("model", "split", "mrr", "ndcg_at_5", "ndcg_at_10", "recall_at_10"))

# 9. Recommendations for a User

`rank_logged_slate` reorders the articles shown in a user's latest validation
impression using this notebook's saved scores. It does not retrieve new articles
from the catalogue and never reads click labels.

In [ ]:
example_user = int(validation_preview["user_code"].iloc[0])
rank_logged_slate(example_user, score_file=final_dir / "validation_scores.parquet", feature_run=RUN, top_n=5)